In [8]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
!pip install better-profanity

In [2]:
import re
import csv
import difflib
import torch

BASE = "grammarly/coedit-large"
ADAPTER = "Shinjan123/coedit-paradetox-lora"

PREFIX = "Clean this sentence: "

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Device: cpu
GPU: None


In [3]:
# word lists (edit these freely)
# Slang: deterministic and explainable. Longest keys are matched first, so "lit af" wins over "af".
SLANG = {"lit af": "really exciting", "tbh": "honestly", "idk": "I do not know", "gonna": "going to",
         "wanna": "want to", "gotta": "have to", "kinda": "kind of", "u": "you", "ur": "your",
         "pls": "please", "thx": "thanks", "cuz": "because", "af": "very", "bro": "", "dude": "",
         "lol": "", "irl": "in real life"}

# Words that sit on the profanity list but are not toxic by themselves (a judgment call: say so in the report).
IGNORE = {"pot", "gay", "omg", "kill"}

# Mild insults the default list misses, so detection catches them.
EXTRA_BAD = ["rascal", "bastard", "idiot", "moron", "clown", "stupid", "jerk"]

# Backstop: used ONLY when the model leaves a bad word behind. Each replacement is milder but keeps the
# criticism (an insult stays a mild criticism, never praise). Crude on purpose: it is a safety net.
SOFTEN = {"idiot": "fool", "idiots": "fools", "moron": "fool", "morons": "fools", "stupid": "unwise",
          "dumb": "unwise", "jerk": "unpleasant person", "bastard": "unpleasant person",
          "rascal": "troublemaker", "clown": "fool", "sucked": "performed poorly", "sucks": "is bad",
          "sucking": "performing poorly", "screwed": "messed", "crap": "nonsense", "damn": "darn",
          "hell": "heck", "shit": "nonsense", "bullshit": "nonsense"}

In [4]:
from better_profanity import profanity

profanity.load_censor_words()

# Add custom words without replacing the default list
profanity.add_censor_words(EXTRA_BAD)
SLANG_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, SLANG), key=len, reverse=True)) + r")\b", re.I)

print("Profanity detector initialized.")

Profanity detector initialized.


In [5]:
# rule-based helpers
def fix_slang(text):
    s = SLANG_RE.sub(lambda m: SLANG[m.group(0).lower()], text)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+([,.!?])", r"\1", s)            # no space before punctuation
    return re.sub(r"^[\s,]+|[\s,]+$", "", s)        # trim commas left where bro/dude was removed

In [6]:
def real_dirty(text):
    """Bad words that remain, ignoring the IGNORE list. Checked word by word so 'assess' is NOT flagged."""
    return [w for w in re.findall(r"[\w']+", text)
            if w.lower() not in IGNORE and profanity.contains_profanity(w)]


def soften_words(text):
    def sub(m):
        w = m.group(0)
        rep = SOFTEN.get(w.lower())
        if not rep:
            return w
        return rep.capitalize() if w[:1].isupper() else rep   # keep the capital letter
    return re.sub(r"[A-Za-z']+", sub, text)


def mask_dirty(text):
    """Last resort: keep the first letter, star the rest (r******)."""
    dirty = {w.lower() for w in real_dirty(text)}
    return re.sub(r"[\w']+", lambda m: m.group(0)[0] + "*" * (len(m.group(0)) - 1)
                  if m.group(0).lower() in dirty else m.group(0), text)


def split_sentences(text):
    return [p for p in re.split(r"(?<=[.!?])\s+", text.strip()) if p]

In [9]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


TOK = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForSeq2SeqLM.from_pretrained(BASE)
model = PeftModel.from_pretrained(base, ADAPTER).to(DEVICE)

model.eval()
print("CleanSpeak model loaded successfully!")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 18.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

CleanSpeak model loaded successfully!


In [10]:
def generate(texts, prefix=PREFIX, beams=4):
    enc = TOK([prefix + t for t in texts], return_tensors="pt", padding=True,
              truncation=True, max_length=96).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**enc, max_length=96, num_beams=beams)
    return TOK.batch_decode(out, skip_special_tokens=True)

def polish(text):
    with model.disable_adapter():                   # adapter off = original CoEdIT, so grammar skills are untouched
        return generate([text], prefix="Fix the grammar: ")[0]

In [11]:
def cleanspeak_sentence(text):
    stages = {"input": text}

    # Stage 1: Slang normalization
    s = fix_slang(text)
    stages["1 slang"] = s
    if not s:
        return "", stages

    # Stage 2: Fine-tuned model
    out = generate([s])[0]
    stages["2 model"] = out

    # Stage 3: Rule-based safety backstop
    if real_dirty(out):
        out = soften_words(out)
        stages["3 backstop"] = out

    # Stage 4: Grammar polishing
    out = polish(out)
    stages["4 polish"] = out

    # Stage 5: Final profanity masking
    if real_dirty(out):
        out = mask_dirty(out)
        stages["5 mask"] = out
    return out, stages


def cleanspeak(text):
    outs = []
    all_stages = []

    for sent in split_sentences(text):
        o, st = cleanspeak_sentence(sent)
        outs.append(o)
        all_stages.append(st)

    return (" ".join(o for o in outs if o), all_stages)

In [12]:
def changes(a, b):
    d = [x for x in difflib.ndiff(a.split(), b.split()) if x[0] in "+-"]
    return "  ".join(d) if d else "(no change)"

In [13]:
def test_cleanspeak(text, verbose=True):
    output, stages = cleanspeak(text)
    flagged = real_dirty(text)

    print("\nINPUT :", text)
    print("FLAGGED WORDS:", flagged if flagged else "-")

    if verbose:
        print("\n--- PIPELINE STAGES ---")
        for st in stages:
            for k, v in st.items():
                print(f"{k:12}: {v}")
            print()

    print("=" * 60)
    print("CLEAN OUTPUT:", output)
    print("CHANGES     :", changes(text, output))
    print("=" * 60)

    return output

In [14]:
test_cleanspeak("Bro, you bastard, why the fuck are you doing this? ")


INPUT : Bro, you bastard, why the fuck are you doing this? 
FLAGGED WORDS: ['bastard', 'fuck']

--- PIPELINE STAGES ---
input       : Bro, you bastard, why the fuck are you doing this?
1 slang     : you bastard, why the fuck are you doing this?
2 model     : Why are you doing this?
4 polish    : Why are you doing this?

CLEAN OUTPUT: Why are you doing this?
CHANGES     : + Why  - Bro,  - you  - bastard,  - why  - the  - fuck


'Why are you doing this?'

In [15]:
test_cleanspeak("tbh bro i kinda wanna go there cuz its lit af")


INPUT : tbh bro i kinda wanna go there cuz its lit af
FLAGGED WORDS: -

--- PIPELINE STAGES ---
input       : tbh bro i kinda wanna go there cuz its lit af
1 slang     : honestly i kind of want to go there because its really exciting
2 model     : honestly i kind of want to go there because its really exciting
4 polish    : Honestly, I kind of want to go there because it's really exciting.

CLEAN OUTPUT: Honestly, I kind of want to go there because it's really exciting.
CHANGES     : + Honestly,  + I  - tbh  - bro  - i  - kinda  + kind  - wanna  + of  + want  + to  - cuz  + because  - its  + it's  - lit  - af  + really  + exciting.


"Honestly, I kind of want to go there because it's really exciting."

In [16]:
# evaluation
TEST_SET = {
    # clean sentences must come back unchanged (apart from case/punctuation): checks for over-editing
    "clean": ["The meeting starts at 10 AM tomorrow.", "I need to assess the class schedule.",
              "Please send me the report by Friday.", "She is going to the library after lunch.",
              "The bus was late, so we missed the first lecture.", "Can you help me with this assignment?",
              "We are meeting at the cafeteria at noon.", "He bought a new laptop yesterday."],
    "grammar": ["She dont like apples", "He go to school every day", "Their going to the park tomorrow.",
                "I seen the movie yesterday and it was realy good",
                "The teacher dont know nothing about this topic", "Me and him was late to class"],
    "slang": ["bro i is gonna go there, its lit af", "idk why u keep ignoring my texts lol",
              "tbh the exam was kinda hard", "pls send me the notes cuz i missed class",
              "dude that presentation was legit amazing", "that happens irl all the time",
              "wanna grab lunch, im starving af"],
    "insult": ["you are a bastard and you never listen", "that rascal stole my bike yesterday",
               "tbh he's a clown, idk why u trust him", "only an idiot would believe that",
               "stop being so stupid and read the instructions", "my manager is a jerk who never says thanks",
               "those morons ruined the whole project", "you dumb kid, give me back my pen"],
    "profanity": ["this is bullshit, the teacher dont know nothing", "what the hell is wrong with you",
                  "shut the fuck up and listen to me", "this damn printer is useless",
                  "get your ass over here right now", "that movie was crap and a waste of money"],
    "mixed": ["bro u r such an idiot, idk why i even talk to u", "this app sucks and ur team is dumb af",
              "omg stop being stupid, its not that hard", "he dont know shit about cars, the dumb jerk"],
}

In [17]:
def norm(s):
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).split()


def retention(src, out, skip):
    """Share of the input's ordinary words that survive. Slang/flagged words are expected to change, so skipped."""
    keep = [w for w in norm(src) if w not in skip]
    o = set(norm(out))
    return sum(w in o for w in keep) / len(keep) if keep else 1.0

In [ ]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

def run_eval(extra_file=None):
    test_set = {k: list(v) for k, v in TEST_SET.items()}
    if extra_file:
        with open(extra_file, encoding="utf-8") as f:
            test_set["custom"] = [ln.strip() for ln in f if ln.strip()]

    slang_words = {w for k in SLANG for w in k.split()}
    rows = []
    for cat, sents in test_set.items():
        for s in sents:
            out, _ = cleanspeak(s)
            skip = slang_words | {w.lower() for w in real_dirty(s)}
            meaning = util.cos_sim(embedder.encode(s), embedder.encode(out)).item() if out else 0.0
            rows.append({"category": cat, "input": s, "output": out,
                         "bad_left": int(bool(real_dirty(out))),
                         "meaning": round(meaning, 3),
                         "retention": round(retention(s, out, skip), 3),
                         "unchanged": int(norm(s) == norm(out))})
            print(f"[{cat}] {s}\n   -> {out}")

    print("\n" + "=" * 78)
    print(f"{'category':10} {'n':>3} {'bad left':>9} {'meaning':>8} {'retention':>10} {'unchanged':>10}")
    for cat in test_set:
        r = [x for x in rows if x["category"] == cat]
        n = len(r)
        print(f"{cat:10} {n:>3} {sum(x['bad_left'] for x in r):>9} "
              f"{sum(x['meaning'] for x in r) / n:>8.2f} {sum(x['retention'] for x in r) / n:>10.2f} "
              f"{(str(sum(x['unchanged'] for x in r)) + '/' + str(n)) if cat == 'clean' else '-':>10}")
    print("Read the insult/mixed outputs by eye too: no metric here can tell 'rude' from a flipped compliment.")

    with open("eval_results_v1.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    print("Saved eval_results_v1.csv")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
results = run_eval()

[clean] The meeting starts at 10 AM tomorrow.
   -> The meeting starts at 10 AM tomorrow.
[clean] I need to assess the class schedule.
   -> I need to assess the class schedule.
[clean] Please send me the report by Friday.
   -> Please send the report by Friday.
[clean] She is going to the library after lunch.
   -> She is going to the library after lunch.
[clean] The bus was late, so we missed the first lecture.
   -> The bus was late, so we missed the first lecture.
[clean] Can you help me with this assignment?
   -> Can you help me with this assignment?
[clean] We are meeting at the cafeteria at noon.
   -> We are meeting at the cafeteria at noon.
[clean] He bought a new laptop yesterday.
   -> He bought a new laptop yesterday.
[grammar] She dont like apples
   -> She doesn't like apples.
[grammar] He go to school every day
   -> He goes to school every day.
[grammar] Their going to the park tomorrow.
   -> They're going to the park tomorrow.
[grammar] I seen the movie yesterday and

In [ ]:
from google.colab import files

files.download("eval_results_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>